# CSI4004 – Text Mining/NLP Techniques
## Smart News Categorization System Using TF-IDF and Machine Learning

**Aim:** Build and evaluate a news-topic classifier and save a trained pipeline for a small Streamlit application.

**Dataset:** scikit-learn's 20 Newsgroups corpus (18,846 documents, 20 original topics).  
**Models:** Multinomial Naive Bayes, Logistic Regression, and Linear Support Vector Classifier.  
**Evaluation:** Accuracy, macro-F1, weighted-F1, per-class precision/recall/F1, confusion matrix, and prediction latency.

> Important scope note: 20 Newsgroups is a collection of discussion posts, not a modern news-wire dataset. Its original labels are 20 newsgroup topics, not exactly Sports/Technology/Business/Politics. This project preserves the original labels to avoid inventing a mapping. If the assignment strictly requires those four news categories, use a dataset with those labels (for example, AG News) and document that dataset change.

## 1. Environment setup

Run this notebook in Google Colab or a local Jupyter environment with internet access for the first dataset download. The dataset is cached by scikit-learn after download.

In [ ]:
%pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

## 2. Imports and reproducibility

In [ ]:
import os, time, json, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.base import clone
import joblib

RANDOM_STATE = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)

## 3. Load the official train/test split

In [ ]:
# Removing headers, signatures and quoted text reduces the chance of learning
# from metadata rather than the message body.
train = fetch_20newsgroups(
    subset="train", shuffle=True, random_state=RANDOM_STATE,
    remove=("headers", "footers", "quotes")
)
test = fetch_20newsgroups(
    subset="test", shuffle=True, random_state=RANDOM_STATE,
    remove=("headers", "footers", "quotes")
)

X_train, y_train = train.data, train.target
X_test, y_test = test.data, test.target
class_names = train.target_names

print(f"Training documents: {len(X_train):,}")
print(f"Test documents: {len(X_test):,}")
print(f"Number of classes: {len(class_names)}")
print(class_names)

## 4. Explore class balance and text length

In [ ]:
class_counts = pd.Series(y_train).map(dict(enumerate(class_names))).value_counts()
display(class_counts.rename_axis("topic").to_frame("training_documents"))

plt.figure(figsize=(11, 6))
class_counts.sort_values().plot(kind="barh")
plt.title("Training documents by original topic")
plt.xlabel("Number of documents")
plt.tight_layout()
plt.show()

lengths = pd.Series([len(t.split()) for t in X_train])
print(lengths.describe().round(1))
plt.figure(figsize=(8, 4))
plt.hist(lengths.clip(upper=1500), bins=40)
plt.title("Training document length (word count; clipped at 1,500)")
plt.xlabel("Words per document")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 5. Preprocessing and feature extraction

The vectorizer lowercases text, tokenizes words, removes English stop words, and builds TF-IDF features. The unigram+bigram model uses single words and adjacent two-word phrases. The vectorizer is placed inside each pipeline so it is fitted on training data only.

In [ ]:
def make_pipeline(model, ngram_range=(1, 2)):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=ngram_range,
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            max_features=200_000
        )),
        ("classifier", model)
    ])

models = {
    "Multinomial Naive Bayes": make_pipeline(MultinomialNB()),
    "Logistic Regression": make_pipeline(
        LogisticRegression(max_iter=2000, C=4.0, random_state=RANDOM_STATE)
    ),
    "Linear SVM": make_pipeline(
        LinearSVC(C=1.0, random_state=RANDOM_STATE)
    )
}

## 6. Train and evaluate models

In [ ]:
results = []
fitted_models = {}
reports = {}

for name, pipeline in models.items():
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start

    start = time.perf_counter()
    predictions = pipeline.predict(X_test)
    predict_seconds = time.perf_counter() - start
    fitted_models[name] = pipeline

    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_test, predictions, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f_weighted, _ = precision_recall_fscore_support(
        y_test, predictions, average="weighted", zero_division=0
    )
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro Precision": p_macro,
        "Macro Recall": r_macro,
        "Macro F1": f_macro,
        "Weighted F1": f_weighted,
        "Training time (s)": fit_seconds,
        "Total test prediction time (s)": predict_seconds,
        "Mean prediction time (ms/doc)": 1000 * predict_seconds / len(X_test)
    })
    reports[name] = classification_report(
        y_test, predictions, target_names=class_names,
        output_dict=True, zero_division=0
    )
    print(f"Completed: {name}")

results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
display(results_df.round(4))
results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

## 7. Compare model performance

In [ ]:
plot_df = results_df.set_index("Model")[["Accuracy", "Macro F1", "Weighted F1"]]
ax = plot_df.plot(kind="bar", figsize=(10, 5), ylim=(0, 1))
ax.set_title("Model comparison on the held-out test set")
ax.set_ylabel("Score")
ax.set_xlabel("")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 8. Detailed report and confusion matrix for the best model

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
best_predictions = best_model.predict(X_test)

print("Selected model by Macro F1:", best_name)
print(classification_report(
    y_test, best_predictions, target_names=class_names, zero_division=0
))

fig, ax = plt.subplots(figsize=(14, 12))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions,
    display_labels=class_names,
    xticks_rotation=90,
    cmap="Blues",
    colorbar=False,
    ax=ax
)
ax.set_title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.show()

## 9. Inspect errors and representative predictions

In [ ]:
error_rows = []
for text, true_id, pred_id in zip(X_test, y_test, best_predictions):
    if true_id != pred_id:
        error_rows.append({
            "true_topic": class_names[true_id],
            "predicted_topic": class_names[pred_id],
            "text_preview": " ".join(text.split())[:350]
        })
errors_df = pd.DataFrame(error_rows)
print(f"Misclassified documents: {len(errors_df):,} / {len(y_test):,}")
display(errors_df.head(10))
errors_df.to_csv(OUTPUT_DIR / "misclassified_examples.csv", index=False)

examples = [
    "NASA announced a new telescope mission to study distant galaxies and planets.",
    "The team scored a goal in the final period and won the hockey game.",
    "The graphics card driver has a bug on my computer system.",
    "The debate in Congress focused on the proposed foreign policy bill."
]
for sample in examples:
    pred = best_model.predict([sample])[0]
    print(f"Predicted topic: {class_names[pred]} | Text: {sample}")

## 10. Prediction latency measurement

The following reports observed inference time for a batch of test documents. Times depend on the runtime environment and should be reported with the hardware/software setup.

In [ ]:
latency_rows = []
sample_texts = X_test[:min(1000, len(X_test))]
for name, pipeline in fitted_models.items():
    start = time.perf_counter()
    _ = pipeline.predict(sample_texts)
    elapsed = time.perf_counter() - start
    latency_rows.append({
        "Model": name,
        "Documents timed": len(sample_texts),
        "Batch time (s)": elapsed,
        "Mean time (ms/document)": elapsed * 1000 / len(sample_texts)
    })
latency_df = pd.DataFrame(latency_rows)
display(latency_df.round(4))
latency_df.to_csv(OUTPUT_DIR / "prediction_latency.csv", index=False)

## 11. Save the best model for the web application

In [ ]:
model_path = OUTPUT_DIR / "news_classifier.joblib"
joblib.dump({
    "pipeline": best_model,
    "class_names": class_names,
    "model_name": best_name
}, model_path)

metadata = {
    "selected_model": best_name,
    "train_documents": len(X_train),
    "test_documents": len(X_test),
    "classes": list(class_names),
    "sklearn_version": sklearn.__version__,
    "selection_metric": "Macro F1",
    "results": results_df.to_dict(orient="records")
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2))
print("Saved model:", model_path)
print("Copy outputs/news_classifier.joblib into the Streamlit app folder.")

## 12. Interpretation checklist for the case study report

After running all cells, use the generated tables and plots to answer:
1. Which model achieved the highest Macro F1?
2. Is the best model also the fastest at prediction?
3. Which original topics have the weakest recall, and what might explain the errors?
4. Do related topics get confused more often than unrelated topics?
5. What limitations arise from the age and genre of the 20 Newsgroups dataset?

Do not insert results into the report until the notebook has been executed successfully. Preserve the actual output tables and figures as evidence.